# Publication package

This notebook retains the supplied scientific processing pipeline and parameter values. Outputs and execution counts have been removed. Local paths are portable, repeated blank cells and duplicate early stack allocations are removed, and explanatory text matches the executed bright/dark filtering and FITS DATE timing. Preview-approval flags start at False: inspect each stage and set its approval flag to True to proceed.

Read `../README.md` and `../docs/parameters.md` first. ROI coordinates are examples from the source S3 configuration and must be set for each image series. No experimental FITS files are distributed. `BM3D_SIGMA=0.3` is retained in transmission units and must be checked on the actual data; it is not 0.3 percent. The observed neighbourhood maximum defines a relative signal scale, not a fully saturated pore-space calibration.


# GDL radiography: FITS → transmission → water change → relative saturation


Use **1-based projection numbers** throughout. No source FITS is overwritten. Run sections in order; rerun a preview as often as needed, then explicitly approve its settings. Changing an upstream setting requires rerunning downstream processing. Every applied step is saved as **float32 TIFF**, with a timestamp manifest, statistics and a representative PNG/PDF. The stacks remain accessible in `stages` without reloading FITS.

**Order:** inspect FITS/time headers → load → preview spike removal → approve dose window / crop → flat/dose normalize → crop → preview/apply BM3D → preview/apply rotation → preview/apply GDL ROI → −ln(T) → subtract dry reference → neighbourhood maximum diagnostics → approve saturation scale → save saturation → independent TIFF viewer.

### What the numbers mean
- Reference formula: `F=(P−mean(DC))/(mean(OB)−mean(DC)+epsilon)`, then `T=F/median(F[dose_window])`. The window must contain stable **open beam**, outside the specimen and wetting region. A changing specimen window would divide out real water uptake.
- `A=−ln(T)` is dimensionless **line-integrated attenuation**, not an attenuation coefficient unless divided by a known path length. `ΔA=A(t)−A(dry)=ln(Tdry/Tt)` increases with added attenuating water. Do not subtract transmission and then take a logarithm.
- `Srelative=100 ΔA / ΔAmax`. The neighbourhood maximum is an **observed reference signal**, not proof of 100% pore saturation. Absolute saturation needs a fully saturated calibration and/or water attenuation, porosity and GDL path length, with scattering/beam-hardening effects assessed. Spatial variation in GDL thickness also affects this normalization.
- Signed values and values above 100% are retained in quantitative TIFF/CSV output; only the displayed colour range is 0–100%. Negative differences may reflect noise, drift or drying. A neighbourhood mean maximum is smaller than some individual pixels, so values above 100% are expected.
- Float32 is preferable to uint16 here: it stores fractional transmission and signed differences without arbitrary rescaling/quantization. NaN marks invalid data, never a fabricated zero. No per-frame min–max normalization is used.
- SD bands describe **spatial variability**, not uncertainty of the mean or independent repeats. The shared dry reference contributes correlated noise. Images must remain aligned across time; one common rotation does not correct drift.

**Colour:** grayscale radiographs; Set3 curves; continuous interpolation of swatch colours sampled from the supplied palette (orange → white → green → blue). Pixel coordinates start at zero; ROI bounds are `(x0,x1,y0,y1)` with exclusive upper bounds. Y increases downward.

In [ ]:
# Run once if packages are missing; then restart the kernel.
# %pip install numpy scipy matplotlib pandas astropy tifffile numba bm3d ipython

In [ ]:
from pathlib import Path
import json, re, warnings, hashlib, platform, importlib.metadata
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Rectangle
from scipy import ndimage as ndi
from astropy.io import fits
from astropy.time import Time, TimeDelta
import tifffile
from IPython.display import display
import warnings
from astropy.io.fits.verify import VerifyWarning

warnings.filterwarnings("ignore", category=VerifyWarning)

# Paths are local to this checkout; environment variables may override them.
import os
REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
INPUT_DIR = Path(os.environ.get('GDL_INPUT_DIR', str(REPO_ROOT / 'data' / 'raw')))
OUTPUT_ROOT = Path(os.environ.get('GDL_OUTPUT_DIR', str(REPO_ROOT / 'outputs')))
PROJ_GLOB, OB_GLOB, DC_GLOB = '*Proj*.fits', '*ob*.fits', '*dc*.fits'

HDU = 0                         # Image extension; primary header is also preserved
PIXEL_SIZE_UM = None             # Set only if known; otherwise all axes use pixels
TIME_KEY = 'DATE'                # Explicit FITS header field; DATE is a file-creation proxy
TIME_FORMAT = 'isot'            # For custom numeric key: 'unix', 'mjd', 'jd', etc.
TIME_SCALE_DEFAULT = 'utc'      # Used only if TIMESYS is absent; recorded in manifest
EXPOSURE_KEY = 'EXPTIME'         # Seconds; None if unavailable
SORT_BY_TIME = True             # First chronological projection becomes dry reference
REFERENCE_PROJECTION = 1
STORAGE = 'ram'                 # 'ram' or 'memmap'; latter uses disk-backed arrays
SAVE_RAW_TIFF = False
FONT_SIZE = 22
plt.rcParams.update({'font.size': FONT_SIZE, 'axes.labelsize': FONT_SIZE,
                     'xtick.labelsize': FONT_SIZE-2, 'ytick.labelsize': FONT_SIZE-2,
                     'legend.fontsize': FONT_SIZE-4, 'lines.linewidth': 2.4,
                     'savefig.dpi': 300, 'pdf.fonttype': 42, 'ps.fonttype': 42})
SET3 = list(plt.get_cmap('Set3').colors)
SAT_COLORS = ['#c07127', '#bd6e24', '#d78c3f', '#d2883e', '#d2893d', '#d5893e', '#e3a764', '#e9ac67', '#f4cf86', '#fdd78b', '#f9d488', '#fbe9a5', '#f9e7a3', '#fceaa6', '#f9e5a2', '#fefac0', '#fff5bb', '#fef8bc', '#ffffff', '#fefefe', '#fefefe', '#fefefe', '#d7fed7', '#d6fed6', '#d2fdd2', '#9ff7b9', '#a0f8ba', '#9ef6b7', '#9cf2b5', '#53bfa1', '#55c5a7', '#6facc9', '#75b4d3', '#71afcd', '#70afcd', '#0076ba', '#0074b7', '#0075b8', '#006dad', '#003559']

SAT_CMAP = LinearSegmentedColormap.from_list('supplied_GDL_palette', SAT_COLORS, N=256)
SAT_CMAP.set_bad('#777777')
RUN = OUTPUT_ROOT / datetime.now().strftime('run_%Y%m%d_%H%M%S_%f')
stages, stage_metadata, provenance = {}, {}, {}

## 1 · Inspect filenames and time headers before loading
Edit patterns above. Groups must be nonempty and disjoint. Exact original FITS headers are archived for every projection, OB and DC. No filesystem modification time is substituted for acquisition time. Set an explicit `TIME_KEY` and `TIME_FORMAT`. This implementation does not auto-select a key or combine DATE-OBS with TIME-OBS. `TIMESYS` is respected through Astropy and timestamps are converted to UTC. Local timestamps need an explicit instrument-specific conversion, not an assumed UTC label.
The supplied configuration reads FITS `DATE` (file creation), not exposure start or midpoint. Elapsed time is the difference between those header timestamps; no half-exposure correction is applied. Missing EXPTIME is stored as 0 only as a flagged compatibility placeholder. Do not interpret that zero as a measured exposure.


In [ ]:
def natural_key(p):
    return [int(v) if v.isdigit() else v.lower() for v in re.split(r'(\d+)', p.name)]

def get_header(p):
    with fits.open(p, memmap=False) as h:
        merged = h[0].header.copy()
        if HDU != 0: merged.extend(h[HDU].header, update=True)
        return merged

def read_image(p):
    a = np.asarray(fits.getdata(p, ext=HDU), dtype=np.float32)
    if a.ndim != 2: raise ValueError(f'{p.name}: expected 2D, got {a.shape}')
    return a

proj_files = sorted(INPUT_DIR.glob(PROJ_GLOB), key=natural_key)
ob_files = sorted(INPUT_DIR.glob(OB_GLOB), key=natural_key)
dc_files = sorted(INPUT_DIR.glob(DC_GLOB), key=natural_key)
for group, paths in [('projections',proj_files),('OB',ob_files),('DC',dc_files)]:
    if not paths: raise ValueError(f'No {group} files: check folder and glob')
    print(group, len(paths), [p.name for p in paths[:3]])
if set(proj_files)&set(ob_files) or set(proj_files)&set(dc_files) or set(ob_files)&set(dc_files):
    raise ValueError('File groups overlap. Make glob patterns more specific.')
h = get_header(proj_files[0])
print('FIRST PROJECTION HEADER — inspect the actual time fields:')
print(repr(h))

In [ ]:
def timestamp_record(h):
    key = TIME_KEY

    if key not in h:
        raise ValueError(
            f"Timestamp field {key!r} is missing from the FITS header."
        )

    raw_value = str(h[key]).strip()

    # Respect TIMESYS when available.
    # Otherwise the configured default is an explicit assumption.
    scale_assumed = 'TIMESYS' not in h
    scale = str(
        h.get('TIMESYS', TIME_SCALE_DEFAULT)
    ).strip().lower()

    if scale not in Time.SCALES:
        raise ValueError(
            f"Unsupported time scale {scale!r}. "
            "Check the detector clock/time-zone settings."
        )

    value = raw_value
    if value.endswith('Z'):
        if scale != 'utc':
            raise ValueError(
                "Timestamp ends in Z (UTC), but the configured time scale "
                f"is {scale!r}."
            )
        value = value[:-1]

    t = Time(value, format=TIME_FORMAT, scale=scale)
    t.precision = 9

    # Preserve exposure information if it exists.
    exposure_value = h.get(EXPOSURE_KEY) if EXPOSURE_KEY else None
    exposure_missing = (
        exposure_value is None
        or str(exposure_value).strip() == ''
    )

    # Zero is a compatibility placeholder, NOT a measured exposure.
    # exposure_missing=True records that distinction.
    exposure = 0.0 if exposure_missing else float(exposure_value)

    if not np.isfinite(exposure) or exposure < 0:
        raise ValueError(
            f"Invalid exposure value: {exposure_value!r}"
        )

    return dict(
        timestamp_utc=t.utc.isot,
        start_utc=None,  # Exposure start cannot be inferred from file creation
        exposure_s=exposure,
        exposure_missing=exposure_missing,
        time_key=key,
        time_raw=raw_value,
        time_scale=scale,
        time_scale_assumed=scale_assumed,
        time_position='file_creation',
        timestamp_basis='FITS DATE: file creation time; acquisition proxy',
    )



    

def alloc(name, shape):
    if STORAGE == 'ram': a=np.empty(shape,np.float32)
    elif STORAGE == 'memmap':
        (RUN/'cache').mkdir(parents=True,exist_ok=True)
        a=np.lib.format.open_memmap(RUN/'cache'/f'{name}.npy',mode='w+',dtype='float32',shape=shape)
    else: raise ValueError('STORAGE must be ram or memmap')
    stages[name]=a
    return a

RUN.mkdir(parents=True,exist_ok=False)
records=[]
for original_order,p in enumerate(proj_files,1):
    records.append(dict(source_file=p.name, source_path=str(p), original_order=original_order,
                        **timestamp_record(get_header(p))))
manifest=pd.DataFrame(records)
if SORT_BY_TIME:
    order=np.argsort(Time(manifest.timestamp_utc.tolist(),scale='utc').tai.jd,kind='stable')
    manifest=manifest.iloc[order].reset_index(drop=True)
proj_files=[Path(p) for p in manifest.source_path]
manifest.insert(0,'projection',np.arange(1,len(manifest)+1))
times=Time(manifest.timestamp_utc.tolist(),scale='utc')
manifest['elapsed_s']=(times-times[0]).to_value('sec')
if (np.diff(manifest.elapsed_s)<0).any(): raise ValueError('Time decreases; use SORT_BY_TIME=True')
if (np.diff(manifest.elapsed_s)==0).any(): warnings.warn('Duplicate timestamps: preserved; inspect manifest')
shape=read_image(proj_files[0]).shape
raw=alloc('00_raw',(len(proj_files),*shape))
for i,p in enumerate(proj_files):
    a=read_image(p)
    if a.shape!=shape: raise ValueError(f'Inconsistent dimensions: {p}')
    raw[i]=a
for group,paths in [('projections',proj_files),('OB',ob_files),('DC',dc_files)]:
    d=RUN/'original_headers'/group; d.mkdir(parents=True,exist_ok=True)
    for i,p in enumerate(paths,1):
        with fits.open(p,memmap=False) as hdus:
            for j,hdu in enumerate(hdus):
                (d/f'{i:06d}_{p.stem}_HDU{j}.txt').write_text(hdu.header.tostring(sep='\n',endcard=True,padding=False))
manifest.to_csv(RUN/'timestamps.csv',index=False)
print('Loaded', raw.shape, 'float32; one full stack uses',round(raw.nbytes/2**30,3),'GiB')
print('Keep enough RAM for multiple stages or restart with STORAGE="memmap".')
display(manifest.head())

## 2 · Shared plotting, saving and ROI helpers
All figures are saved as PNG and vector PDF. Each stage has its own TIFFs, `timestamps.csv`, `statistics.csv`, and processing metadata in TIFF ImageDescription. Full original headers remain in the run folder. Reapplying a step replaces that step within this run; start a new run for a separate parameter experiment. TIFF names use 1-based projection numbers; the manifest also retains source filenames.

In [ ]:
def check_box(box, shape):
    h,w=shape
    if box is None: return (0,w,0,h)
    if len(box)!=4 or any(int(v)!=v for v in box): raise ValueError('Use four integer bounds')
    x0,x1,y0,y1=map(int,box)
    if not (0<=x0<x1<=w and 0<=y0<y1<=h): raise ValueError(f'Box {box} outside shape {shape}')
    return (x0,x1,y0,y1)

def crop(a,box):
    x0,x1,y0,y1=check_box(box,a.shape[-2:]); return a[...,y0:y1,x0:x1]

def selected(stage,projection):
    if not isinstance(projection,(int,np.integer)) or not 1<=projection<=len(stages[stage]):
        raise ValueError('Projection number must be an integer from 1 to N')
    return stages[stage][projection-1]

def limits(a):
    v=np.asarray(a)[np.isfinite(a)]
    if not len(v): return (0,1)
    lo,hi=np.percentile(v,[1,99]); return (lo,hi if hi>lo else lo+1)

def image_panel(ax,a,title,box=None,grid=False,vlim=None,cmap='gray'):
    lo,hi=limits(a) if vlim is None else vlim
    m=ax.imshow(a,cmap=cmap,vmin=lo,vmax=hi,origin='upper',interpolation='nearest')
    ax.set(xlabel='X (pixel)',ylabel='Y (pixel)',title=title)
    if box is not None:
        x0,x1,y0,y1=check_box(box,a.shape)
        ax.add_patch(Rectangle((x0-.5,y0-.5),x1-x0,y1-y0,fill=False,ec='#d95f02',lw=2))
    if grid: ax.grid(alpha=.45,linestyle=':',color='#377eb8')
    return m

def savefig(fig,name):
    d=RUN/'figures'; d.mkdir(exist_ok=True)
    for ext in ['png','pdf']: fig.savefig(d/f'{name}.{ext}',bbox_inches='tight')
    plt.show()

def stats(a):
    v=a[np.isfinite(a)]
    return dict(mean=float(np.mean(v)) if len(v) else np.nan,
                sd=float(np.std(v,ddof=0)) if len(v) else np.nan,
                median=float(np.median(v)) if len(v) else np.nan,
                p01=float(np.percentile(v,1)) if len(v) else np.nan,
                p99=float(np.percentile(v,99)) if len(v) else np.nan,
                minimum=float(np.min(v)) if len(v) else np.nan,
                maximum=float(np.max(v)) if len(v) else np.nan,
                valid_pixels=int(len(v)),valid_fraction=float(len(v)/a.size))

def save_stage(name,details,unit='a.u.'):
    a=stages[name]; d=RUN/name; d.mkdir(exist_ok=True)
    stage_metadata[name]=dict(details=details,unit=unit)
    table=manifest.copy(); rows=[]
    for i,frame in enumerate(a):
        fname=f'proj_{i+1:06d}.tif'
        meta=dict(manifest.iloc[i].to_dict(),stage=name,unit=unit,details=details,
                  upstream={k:v for k,v in stage_metadata.items() if k<name},pixel_size_um=PIXEL_SIZE_UM,
                  full_original_headers='../original_headers',invalid='NaN')
        tifffile.imwrite(d/fname,np.asarray(frame,np.float32),photometric='minisblack',
                         description=json.dumps(meta,default=str),metadata=None)
        rows.append(stats(frame))
    table['tiff_file']=[f'proj_{i+1:06d}.tif' for i in range(len(a))]
    table.to_csv(d/'timestamps.csv',index=False)
    table=pd.concat([table,pd.DataFrame(rows)],axis=1)
    table.to_csv(d/'statistics.csv',index=False)
    (RUN/'processing.json').write_text(json.dumps(stage_metadata,indent=2,default=str))
    fig,ax=plt.subplots(figsize=(9,7))
    im=image_panel(ax,a[0],f'{name} · projection 1',cmap=SAT_CMAP if unit=='%' else 'gray',
                   vlim=(0,100) if unit=='%' else None)
    fig.colorbar(im,ax=ax,label=unit); savefig(fig,name+'_example')
    print('Saved', name, a.shape)
    return table

# if SAVE_RAW_TIFF: save_stage('00_raw',{'source':'original FITS'},'detector counts')
versions={p:importlib.metadata.version(p) for p in ['numpy','scipy','pandas','matplotlib','astropy','tifffile']}
(RUN/'software_versions.json').write_text(json.dumps(dict(python=platform.python_version(),**versions),indent=2))

## 3 · Selective bright- and dark-spot removal (preview)
A local median supplies a replacement **only for detected positive or negative outliers**. The threshold is local median absolute residual × 1.4826 × `SPIKE_SIGMA`, with a global residual-noise floor; small connected components only are replaced. OB and DC frames receive the same treatment before averaging. This is safer for fine GDL structure than filtering every pixel, but real tiny bright features can still be mistaken for spots: inspect the mask. Large spots need a larger odd window than the spot diameter. Lower sigma is more aggressive. No temporal median is applied, so real rapid uptake is preserved.

In [ ]:
PREVIEW_PROJECTION = 1

SPIKE_WINDOW = 7             # Odd number; larger than the spots
SPIKE_SIGMA = 2             # Lower = more aggressive detection
SPIKE_MAX_AREA = 300          # Maximum connected spot area, in pixels
SPIKE_ABSOLUTE_FLOOR = 0.0    # Minimum intensity difference, in detector counts
ENABLE_DESPIKE = True


def despike(a):
    if SPIKE_WINDOW < 3 or SPIKE_WINDOW % 2 != 1:
        raise ValueError("SPIKE_WINDOW must be odd and >= 3")

    if SPIKE_SIGMA <= 0 or SPIKE_MAX_AREA < 1:
        raise ValueError("Invalid spike settings")

    if SPIKE_ABSOLUTE_FLOOR < 0:
        raise ValueError("SPIKE_ABSOLUTE_FLOOR must be >= 0")

    a = np.asarray(a, dtype=np.float32)
    finite = np.isfinite(a)

    if not finite.any():
        raise ValueError("Image contains no finite pixels")

    if not ENABLE_DESPIKE:
        return a.copy(), np.zeros_like(finite)

    # Fill invalid pixels temporarily using the nearest valid pixel.
    # Original invalid pixels remain unchanged in the returned image.
    if finite.all():
        z = a
    else:
        indices = ndi.distance_transform_edt(
            ~finite,
            return_distances=False,
            return_indices=True,
        )
        z = a[tuple(indices)]

    # Local background estimate.
    med = ndi.median_filter(
        z, size=SPIKE_WINDOW, mode="reflect"
    )

    residual = z - med

    # Robust estimates of local and global residual variation.
    local_noise = 1.4826 * ndi.median_filter(
        np.abs(residual),
        size=SPIKE_WINDOW,
        mode="reflect",
    )

    valid_residual = residual[finite]
    global_noise = 1.4826 * np.median(
        np.abs(valid_residual - np.median(valid_residual))
    )

    threshold = np.maximum(
        SPIKE_SIGMA * np.maximum(local_noise, global_noise),
        SPIKE_ABSOLUTE_FLOOR,
    )

    # Detect bright and dark spots separately.
    bright_candidates = finite & (residual > threshold)
    dark_candidates = finite & (residual < -threshold)

    def keep_small_components(candidates):
        labels, _ = ndi.label(
            candidates,
            structure=np.ones((3, 3), dtype=bool),
        )
        areas = np.bincount(labels.ravel())

        return (
            candidates
            & (areas[labels] <= SPIKE_MAX_AREA)
        )

    bright_mask = keep_small_components(bright_candidates)
    dark_mask = keep_small_components(dark_candidates)
    mask = bright_mask | dark_mask

    # Replace only detected pixels; preserve all other values.
    result = a.copy()
    result[mask] = med[mask]

    return result, mask



a=selected('00_raw',PREVIEW_PROJECTION); b,mask=despike(a)
fig,axs=plt.subplots(1,3,figsize=(19,6),layout='constrained')
for ax,arr,title in zip(axs,[a,b,mask],['Original','Selective median','Replaced pixels']):
    image_panel(ax,arr,title,vlim=limits(a) if title!='Replaced pixels' else (0,1))
savefig(fig,'spike_preview'); print('Replaced pixels:',mask.sum(), '/',mask.size)
SPIKE_PREVIEW_SETTINGS=(SPIKE_WINDOW,SPIKE_SIGMA,SPIKE_MAX_AREA,SPIKE_ABSOLUTE_FLOOR,ENABLE_DESPIKE)

## 4 · Dose window and initial crop preview (original detector coordinates)
Set `DOSE_ROI` to open beam; `CROP_BOX=None` keeps the full frame. The dose window can lie outside the crop because dose normalization happens **before** cropping. Rerun this preview until both boxes are correct.

In [ ]:
DOSE_ROI = (800,1000,1930,2000)      # Example from the reference notebook — adapt to your images
CROP_BOX = (950,1170,300,2000)                # e.g. (100,1500,50,1800)
a=selected('00_raw',PREVIEW_PROJECTION)
DOSE_ROI=check_box(DOSE_ROI,a.shape); CROP_BOX=check_box(CROP_BOX,a.shape)
fig,axs=plt.subplots(1,2,figsize=(15,7),layout='constrained')
image_panel(axs[0],a,'Dose reference in original coordinates',DOSE_ROI,grid=True)
image_panel(axs[1],a,'Initial crop in original coordinates',CROP_BOX,grid=True)
savefig(fig,'dose_crop_preview')
CROP_PREVIEW_SETTINGS=(DOSE_ROI,CROP_BOX)

In [ ]:
import os
import time
import hashlib
from concurrent.futures import ThreadPoolExecutor
import numpy as np
from scipy import ndimage as ndi

CPU_THREADS = min(4, os.cpu_count() or 1)  # Start with 4; memory use grows with concurrency
REUSE_CALIBRATION = True                  # Reuse only matching in-memory calibration

try:
    import numba as nb
except ImportError as exc:
    raise ImportError('Run %pip install numba in a separate cell, then rerun this cell.') from exc

nb.set_num_threads(min(CPU_THREADS, nb.config.NUMBA_NUM_THREADS))

# Keep this compiled function when rerunning this setup cell in the same kernel.
if '_gdl_histogram_median_v1' not in globals():
    @nb.njit(parallel=True)
    def _gdl_histogram_median_v1(padded, height, width, window):
        output = np.empty((height, width), np.uint16)
        target = (window * window) // 2
        for y in nb.prange(height):
            fine = np.zeros(65536, np.int32)
            coarse = np.zeros(256, np.int32)
            for yy in range(y, y + window):
                for xx in range(window):
                    v = int(padded[yy, xx])
                    fine[v] += 1
                    coarse[v >> 8] += 1
            for x in range(width):
                if x > 0:
                    for yy in range(y, y + window):
                        old = int(padded[yy, x - 1])
                        new = int(padded[yy, x + window - 1])
                        fine[old] -= 1
                        coarse[old >> 8] -= 1
                        fine[new] += 1
                        coarse[new >> 8] += 1
                below = 0
                block = 0
                while below + coarse[block] <= target:
                    below += coarse[block]
                    block += 1
                value = block << 8
                while below + fine[value] <= target:
                    below += fine[value]
                    value += 1
                output[y, x] = value
        return output

_backend_counts = {'histogram': 0, 'scipy_float': 0}

def fast_median_exact(a, window):
    """Exact square median, SciPy reflect boundaries; no intensity quantization."""
    lo = float(np.min(a))
    hi = float(np.max(a))
    # Most raw 16-bit radiographs are integer counts even when loaded as float32.
    if (np.isfinite(lo) and np.isfinite(hi) and hi - lo <= 65535
            and np.all(a == np.floor(a))):
        shifted = (a - lo).astype(np.uint16)
        radius = window // 2
        # numpy symmetric == scipy ndimage reflect (half-sample symmetric).
        padded = np.pad(shifted, radius, mode='symmetric')
        result = _gdl_histogram_median_v1(padded, *a.shape, window)
        _backend_counts['histogram'] += 1
        return (result.astype(a.dtype) + lo).astype(a.dtype)

    # Genuine floating-point / wider-range data: exact SciPy median on overlapping
    # row strips. Halos preserve the full 2D median, including strip boundaries.
    _backend_counts['scipy_float'] += 1
    workers = min(CPU_THREADS, a.shape[0])
    if workers <= 1:
        return ndi.median_filter(a, size=window, mode='reflect')
    radius = window // 2
    bounds = np.linspace(0, a.shape[0], workers + 1, dtype=int)
    out = np.empty_like(a)
    def strip(j):
        start, stop = int(bounds[j]), int(bounds[j + 1])
        top, bottom = max(0, start-radius), min(a.shape[0], stop+radius)
        filtered = ndi.median_filter(a[top:bottom], size=window, mode='reflect')
        out[start:stop] = filtered[start-top:stop-top]
    with ThreadPoolExecutor(max_workers=workers) as pool:
        list(pool.map(strip, range(workers)))
    return out


def despike(a):
    """Same bright/dark algorithm and settings as the preceding supplied function."""
    if SPIKE_WINDOW < 3 or SPIKE_WINDOW % 2 != 1:
        raise ValueError('SPIKE_WINDOW must be odd and >=3')
    if SPIKE_SIGMA <= 0 or SPIKE_MAX_AREA < 1 or SPIKE_ABSOLUTE_FLOOR < 0:
        raise ValueError('Invalid spike settings')
    a = np.asarray(a, dtype=np.float32)
    finite = np.isfinite(a)
    if not finite.any():
        raise ValueError('Image contains no finite pixels')
    if not ENABLE_DESPIKE:
        return a.copy(), np.zeros_like(finite)
    if finite.all():
        z = a
    else:
        indices = ndi.distance_transform_edt(
            ~finite, return_distances=False, return_indices=True)
        z = a[tuple(indices)]
    med = fast_median_exact(z, SPIKE_WINDOW)
    residual = z - med
    local_noise = 1.4826 * fast_median_exact(np.abs(residual), SPIKE_WINDOW)
    valid_residual = residual[finite]
    global_noise = 1.4826 * np.median(
        np.abs(valid_residual - np.median(valid_residual)))
    threshold = np.maximum(
        SPIKE_SIGMA * np.maximum(local_noise, global_noise),
        SPIKE_ABSOLUTE_FLOOR)
    def keep(candidates):
        labels, _ = ndi.label(candidates, structure=np.ones((3,3), dtype=bool))
        areas = np.bincount(labels.ravel())
        return candidates & (areas[labels] <= SPIKE_MAX_AREA)
    mask = keep(finite & (residual > threshold)) | keep(finite & (residual < -threshold))
    result = a.copy()
    result[mask] = med[mask]
    return result, mask

# First call compiles once; subsequent frames do not pay this compilation cost.
print('Preparing exact median accelerator (first run compiles the kernel)...', flush=True)
_gdl_histogram_median_v1(np.zeros((5,5), np.uint16), 3, 3, 3)
print(f'Ready: {nb.get_num_threads()} median threads. Bright/dark thresholds unchanged.')
FAST_DESPIKE_VERSION = 'exact_bright_dark_histogram_v1'
_fast_setup_signature = (FAST_DESPIKE_VERSION, id(despike))


# REPLACEMENT CELL 1 — calibration. Run FAST SETUP before this cell.
APPROVE_NORMALIZATION = False
if not APPROVE_NORMALIZATION:
    raise RuntimeError('Inspect previews, then set APPROVE_NORMALIZATION=True')
if SPIKE_PREVIEW_SETTINGS != (SPIKE_WINDOW, SPIKE_SIGMA, SPIKE_MAX_AREA,
                              SPIKE_ABSOLUTE_FLOOR, ENABLE_DESPIKE):
    raise RuntimeError('Spike settings changed: rerun preview, then FAST SETUP')
if CROP_PREVIEW_SETTINGS != (DOSE_ROI, CROP_BOX):
    raise RuntimeError('Rerun dose/crop preview')
if globals().get('_fast_setup_signature') != (FAST_DESPIKE_VERSION, id(despike)):
    raise RuntimeError('The preview cell redefined despike: rerun FAST SETUP immediately before this cell')

cal_started = time.perf_counter()

def calibration_mean(paths, label):
    total = np.zeros(shape, np.float64)
    count = np.zeros(shape, np.int32)
    for j, p in enumerate(paths, 1):
        started = time.perf_counter()
        a = read_image(p)
        if a.shape != shape:
            raise ValueError(f'Calibration shape mismatch: {p}')
        a, _ = despike(a)
        ok = np.isfinite(a)
        # where= avoids allocating a full np.where result per image.
        np.add(total, a, out=total, where=ok)
        count += ok
        print(f'{label} {j}/{len(paths)} | {p.name} | '
              f'{time.perf_counter()-started:.1f} s', flush=True)
    return np.divide(total, count, out=np.full(shape, np.nan),
                     where=count>0).astype(np.float32)

# Count-based normalization still requires matching integration times.
if EXPOSURE_KEY is not None:
    cal_exp = [get_header(p).get(EXPOSURE_KEY) for p in ob_files + dc_files]
    known = ([float(v) for v in cal_exp if v is not None]
             + manifest.loc[~manifest.exposure_missing, 'exposure_s'].tolist())
    if known and not np.allclose(known, known[0], rtol=1e-5, atol=1e-8):
        raise ValueError('Exposure times differ across OB/DC/projections. Use matching calibration frames; count-rate correction needs matched dark treatment.')
    if any(v is None for v in cal_exp):
        warnings.warn('Calibration exposure metadata missing: verify matching integration times manually')

# Reuse only if sources, file size/mtime, dimensions, HDU and cleaning settings match.
# Set REUSE_CALIBRATION=False if files were altered without updating their timestamps.
def file_signature(paths):
    return tuple((str(p.resolve()), p.stat().st_size, p.stat().st_mtime_ns) for p in paths)

calibration_signature = (
    FAST_DESPIKE_VERSION, tuple(shape), globals().get('HDU', 0),
    SPIKE_WINDOW, SPIKE_SIGMA, SPIKE_MAX_AREA, SPIKE_ABSOLUTE_FLOOR, ENABLE_DESPIKE,
    file_signature(dc_files), file_signature(ob_files),
)
cache = globals().get('_gdl_calibration_cache')
if REUSE_CALIBRATION and cache is not None and cache['signature'] == calibration_signature:
    dc, ob = cache['dc'], cache['ob']
    print('Reusing matching cleaned DC/OB averages from memory.', flush=True)
else:
    dc = calibration_mean(dc_files, 'DC')
    ob = calibration_mean(ob_files, 'OB')
    _gdl_calibration_cache = dict(signature=calibration_signature, dc=dc, ob=ob)

effective = ob - dc
positive = effective[np.isfinite(effective) & (effective > 0)]
if not positive.size:
    raise ValueError('No positive effective open beam')
epsilon = 1e-6 * np.median(positive)
denominator = np.where(np.isfinite(effective) & (effective > epsilon),
                       effective + epsilon, np.nan)
caldir = RUN / 'calibration'
caldir.mkdir(parents=True, exist_ok=True)
for name, a in [('mean_DC', dc), ('mean_OB', ob), ('effective_OB', effective)]:
    tifffile.imwrite(caldir / f'{name}.tif', a.astype(np.float32, copy=False),
                     photometric='minisblack', metadata=None)
print(f'Calibration finished in {time.perf_counter()-cal_started:.1f} s. '
      f'Median passes by backend: {_backend_counts}', flush=True)
_normalization_calibration_signature = calibration_signature


In [ ]:
# REPLACEMENT CELL 2 — normalization and export, in one pass.
# Keeps all four full/cropped stages available and saves every TIFF and CSV.
TIFF_WRITERS = 2                # Use 1 on a slow/network disk; 2 often helps on SSD
PRINT_EVERY = 5                 # Always also prints the first and last projection



# Stacks are allocated once below, after obsolete references are released.




if TIFF_WRITERS not in (1, 2, 3, 4) or PRINT_EVERY < 1:
    raise ValueError('Use 1–4 TIFF writers and PRINT_EVERY >=1')
if globals().get('_fast_setup_signature') != (FAST_DESPIKE_VERSION, id(despike)):
    raise RuntimeError('Run FAST SETUP and replacement calibration cell first')
current_signature = (
    FAST_DESPIKE_VERSION, tuple(shape), globals().get('HDU', 0),
    SPIKE_WINDOW, SPIKE_SIGMA, SPIKE_MAX_AREA, SPIKE_ABSOLUTE_FLOOR, ENABLE_DESPIKE,
    file_signature(dc_files), file_signature(ob_files),
)
if globals().get('_normalization_calibration_signature') != current_signature:
    raise RuntimeError('Calibration/settings changed: rerun replacement calibration cell')
if CROP_PREVIEW_SETTINGS != (DOSE_ROI, CROP_BOX):
    raise RuntimeError('Rerun dose/crop preview first')

started_all = time.perf_counter()
_backend_counts.update(histogram=0, scipy_float=0)
# Drop obsolete output references before allocating replacement arrays.
# With memmap, this closes notebook-owned references before opening the same .npy paths.
for _name in ['01_despiked', '02_flat_field', '03_dose_normalized', '04_cropped']:
    stages.pop(_name, None)
for _var in ['clean', 'flat', 'dose', 'cropped']:
    globals().pop(_var, None)
clean = alloc('01_despiked', raw.shape)
flat = alloc('02_flat_field', raw.shape)
dose = alloc('03_dose_normalized', raw.shape)
cropped = alloc('04_cropped', (len(raw), *crop(raw[0], CROP_BOX).shape))

settings = dict(spike_window=SPIKE_WINDOW, sigma=SPIKE_SIGMA,
                max_area=SPIKE_MAX_AREA, absolute_floor=SPIKE_ABSOLUTE_FLOOR,
                enabled=ENABLE_DESPIKE, polarity='bright and dark',
                median_implementation=FAST_DESPIKE_VERSION,
                integer_fast_path='exact histogram; no quantization',
                float_fallback='exact parallel SciPy median')
output_specs = [
    ('01_despiked', settings, 'detector counts'),
    ('02_flat_field', dict(formula='(P-DC)/(OB-DC+epsilon)', epsilon=float(epsilon),
        ob_files=[str(p) for p in ob_files], dc_files=[str(p) for p in dc_files]), 'transmission'),
    ('03_dose_normalized', dict(dose_roi=DOSE_ROI, estimator='median',
        invalid='NaN; no zero fallback'), 'transmission'),
    ('04_cropped', dict(crop_box=CROP_BOX, coordinate_system='original detector'), 'transmission'),
]
for name, details, unit in output_specs:
    (RUN / name).mkdir(exist_ok=True)
    stage_metadata[name] = dict(details=details, unit=unit)

# One finite-value extraction and one multi-quantile pass per frame,
# instead of separate full-image percentile passes for p01/p99.
def quick_stats(a):
    v = a[np.isfinite(a)]
    if not v.size:
        return dict(mean=np.nan, sd=np.nan, median=np.nan, p01=np.nan, p99=np.nan,
                    minimum=np.nan, maximum=np.nan, valid_pixels=0, valid_fraction=0.0)
    result = dict(mean=float(np.mean(v)), sd=float(np.std(v, ddof=0)),
                  minimum=float(np.min(v)), maximum=float(np.max(v)),
                  valid_pixels=int(v.size), valid_fraction=float(v.size / a.size))
    # v is a detached temporary, so partitioning it cannot change the saved image.
    q = np.percentile(v, [1, 50, 99], overwrite_input=True)
    result.update(p01=float(q[0]), median=float(np.asarray(q[1], dtype=v.dtype)), p99=float(q[2]))
    return result

metadata_bases = {}
for name, details, unit in output_specs:
    metadata_bases[name] = dict(
        stage=name, unit=unit, details=details,
        upstream={k:v for k,v in stage_metadata.items() if k<name},
        pixel_size_um=PIXEL_SIZE_UM, full_original_headers='../original_headers', invalid='NaN')

# Writers read completed, disjoint frame slots; the processing loop never changes
# a submitted slot. Futures are bounded to prevent a backlog of pending work.
def export_frame(i):
    tick = time.perf_counter()
    record = manifest.iloc[i].to_dict()
    fname = f'proj_{i+1:06d}.tif'
    rows = {}
    for name, _, _ in output_specs:
        frame = stages[name][i]
        meta = dict(record, **metadata_bases[name])
        tifffile.imwrite(RUN / name / fname, frame, photometric='minisblack',
                         description=json.dumps(meta, default=str), metadata=None)
        rows[name] = quick_stats(frame)
    return i, rows, time.perf_counter()-tick

stage_rows = {name: [None]*len(raw) for name,_,_ in output_specs}
export_seconds = np.zeros(len(raw), np.float64)

def collect(future):
    i, rows, seconds = future.result()  # Propagate write failures immediately.
    for name, row in rows.items():
        stage_rows[name][i] = row
    export_seconds[i] = seconds

qc = []
pending = []
with ThreadPoolExecutor(max_workers=TIFF_WRITERS) as pool:
    for i, a in enumerate(raw):
        tick = time.perf_counter()
        clean[i], mask = despike(a)
        np.subtract(clean[i], dc, out=flat[i])
        np.divide(flat[i], denominator, out=flat[i])
        window = crop(flat[i], DOSE_ROI)
        valid = np.isfinite(window)
        if valid.mean() < 0.9:
            raise ValueError(f'Projection {i+1}: <90% valid dose window')
        factor = float(np.nanmedian(window))
        if not np.isfinite(factor) or factor <= 0:
            raise ValueError(f'Projection {i+1}: invalid dose; fix window')
        np.divide(flat[i], factor, out=dose[i])
        cropped[i] = crop(dose[i], CROP_BOX)
        process_seconds = time.perf_counter()-tick
        qc.append(dict(projection=i+1, dose_factor=factor,
            dose_sd=float(np.nanstd(window)), spike_count=int(mask.sum()),
            spike_fraction=float(mask.mean()),
            nonpositive_fraction=float(np.mean(dose[i]<=0)),
            processing_s=process_seconds))
        if len(pending) >= TIFF_WRITERS:
            collect(pending.pop(0))
        pending.append(pool.submit(export_frame, i))
        if i==0 or (i+1)%PRINT_EVERY==0 or i==len(raw)-1:
            elapsed = time.perf_counter()-started_all
            eta = elapsed/(i+1)*(len(raw)-i-1)
            print(f'Processed {i+1}/{len(raw)} | frame {process_seconds:.2f} s | '
                  f'elapsed {elapsed/60:.1f} min | remaining ~{eta/60:.1f} min '
                  '(plus final writes)', flush=True)
    for future in pending:
        collect(future)

# Flush disk-backed arrays only after all workers have finished reading.
for name, _, _ in output_specs:
    if isinstance(stages[name], np.memmap):
        stages[name].flush()

normalization_qc = manifest.merge(pd.DataFrame(qc), on='projection')
normalization_qc['export_and_statistics_s'] = export_seconds
normalization_qc.to_csv(RUN / 'normalization_QC.csv', index=False)
for name, _, _ in output_specs:
    table = manifest.copy()
    table['tiff_file'] = [f'proj_{i+1:06d}.tif' for i in range(len(raw))]
    table.to_csv(RUN / name / 'timestamps.csv', index=False)
    pd.concat([table.reset_index(drop=True), pd.DataFrame(stage_rows[name])], axis=1).to_csv(
        RUN / name / 'statistics.csv', index=False)
(RUN / 'processing.json').write_text(json.dumps(stage_metadata, indent=2, default=str))
(RUN / 'normalization_performance.json').write_text(json.dumps(dict(
    seconds_processing_and_export=time.perf_counter()-started_all,
    cpu_threads=CPU_THREADS, tiff_writers=TIFF_WRITERS,
    median_backend_counts=_backend_counts, numba_version=nb.__version__), indent=2))

# Preserve the same representative stage plots without re-exporting the stacks.
for name, _, unit in output_specs:
    fig, ax = plt.subplots(figsize=(9,7))
    im = image_panel(ax, stages[name][0], f'{name} · projection 1')
    fig.colorbar(im, ax=ax, label=unit)
    savefig(fig, name+'_example')

fig, axs = plt.subplots(2,1, figsize=(11,9), sharex=True, layout='constrained')
axs[0].plot(manifest.elapsed_s, normalization_qc.dose_factor, color=SET3[3])
axs[0].set_ylabel('Dose factor')
axs[1].plot(manifest.elapsed_s, normalization_qc.spike_fraction*100, color=SET3[4])
axs[1].set(ylabel='Replaced pixels (%)', xlabel='Elapsed time (s)')
savefig(fig, 'normalization_QC')
print(f'Finished: all four stages and timestamps saved. Total '
      f'{(time.perf_counter()-started_all)/60:.2f} min. Median passes: {_backend_counts}', flush=True)
print('Continue with your existing BM3D cell; cropped and stages are ready.')


## 5 · BM3D preview and apply
BM3D runs on each cropped transmission image independently. `BM3D_SIGMA` is noise SD **in transmission units** (0.005 means 0.5 percentage points), never 0–255 units. Tune on a representative image and inspect removed structure in the residual. A spatially uniform sigma approximates the noise; flat-field ratios may have nonuniform noise. No per-image rescaling or clipping is used. Invalid pixels are temporarily filled from nearest valid neighbours for the filter, then restored to NaN. Large invalid regions should be cropped away. Set `ENABLE_BM3D=False` to bypass. BM3D is usually the slowest step.

In [ ]:
# !pip install bm3d

In [ ]:
ENABLE_BM3D = True
BM3D_SIGMA = 0.3
PREVIEW_PROJECTION = 1

def denoise(a):
    if not ENABLE_BM3D: return a.copy()
    from bm3d import bm3d, BM3DStages
    if BM3D_SIGMA<=0: raise ValueError('BM3D_SIGMA must be positive')
    valid=np.isfinite(a)
    if not valid.any(): raise ValueError('No finite pixels for BM3D')
    indices=ndi.distance_transform_edt(~valid,return_distances=False,return_indices=True)
    z=a[tuple(indices)] if not valid.all() else a
    result=np.asarray(bm3d(z.astype(np.float64),sigma_psd=BM3D_SIGMA,
                          stage_arg=BM3DStages.ALL_STAGES),np.float32)
    result[~valid]=np.nan
    return result

a=selected('04_cropped',PREVIEW_PROJECTION); b=denoise(a)
fig,axs=plt.subplots(1,3,figsize=(19,6),layout='constrained')
for ax,arr,title in zip(axs,[a,b,a-b],['Before BM3D','After BM3D','Removed signal (before − after)']):
    m=image_panel(ax,arr,title,vlim=limits(a) if title!='Removed signal (before − after)' else None)
    fig.colorbar(m,ax=ax,shrink=.7)
savefig(fig,'BM3D_preview')
print('Mean removed signal:',float(np.nanmean(a-b)),'; SD:',float(np.nanstd(a-b)))
BM3D_PREVIEW_SETTINGS=(ENABLE_BM3D,BM3D_SIGMA)

In [ ]:
APPROVE_BM3D = False
if not APPROVE_BM3D: raise RuntimeError('Inspect BM3D preview, then approve')
if BM3D_PREVIEW_SETTINGS!=(ENABLE_BM3D,BM3D_SIGMA): raise RuntimeError('Rerun BM3D preview')
den=alloc('05_BM3D',cropped.shape)
residuals=[]
for i,a in enumerate(cropped):
    den[i]=denoise(a)
    residuals.append(dict(projection=i+1,**stats(a-den[i])))
    if (i+1)%10==0 or i==len(cropped)-1: print('BM3D',i+1,'/',len(cropped))
manifest.merge(pd.DataFrame(residuals),on='projection').to_csv(RUN/'BM3D_residual_statistics.csv',index=False)
save_stage('05_BM3D',dict(enabled=ENABLE_BM3D,sigma_transmission=BM3D_SIGMA,
                         version=importlib.metadata.version('bm3d') if ENABLE_BM3D else None),'transmission')

## 6 · Rotation preview with vertical ruler
Type an angle in degrees; positive rotates counterclockwise on the displayed image. The canvas keeps the same size; corners outside the detector become NaN. Bilinear interpolation avoids high-order overshoot. The ruler is drawn only on the plot. Edit angle/ruler/projection and rerun this cell; the next cell applies the approved angle to all projections. Coordinates here are relative to the initial crop.

In [ ]:
ROTATION_DEG = 1.0
RULER_X = None                  # None = image centre; otherwise choose integer x
PREVIEW_PROJECTION = 1

def rotate_image(a,angle):
    if not np.isfinite(angle): raise ValueError('Angle must be finite')
    if angle==0: return a.copy()
    valid=np.isfinite(a)
    value=ndi.rotate(np.where(valid,a,0),angle,reshape=False,order=1,mode='constant',cval=0,prefilter=False)
    weight=ndi.rotate(valid.astype(np.float32),angle,reshape=False,order=1,mode='constant',cval=0,prefilter=False)
    value[weight<0.99999]=np.nan
    return value.astype(np.float32)

a=selected('05_BM3D',PREVIEW_PROJECTION); b=rotate_image(a,ROTATION_DEG)
x=a.shape[1]//2 if RULER_X is None else RULER_X
if not 0<=x<a.shape[1]: raise ValueError('Ruler outside image')
fig,axs=plt.subplots(1,2,figsize=(16,8),layout='constrained')
for ax,arr,title in zip(axs,[a,b],['Before rotation',f'After {ROTATION_DEG:g}°']):
    image_panel(ax,arr,title,grid=True,vlim=limits(a)); ax.axvline(x,color='#e41a1c',ls='--',lw=1.8)
savefig(fig,'rotation_preview')
ROTATION_PREVIEW_SETTINGS=ROTATION_DEG

In [ ]:
APPROVE_ROTATION = False
if not APPROVE_ROTATION: raise RuntimeError('Inspect rotation preview, then approve')
if ROTATION_PREVIEW_SETTINGS!=ROTATION_DEG: raise RuntimeError('Angle changed: rerun preview')
rotated=alloc('06_rotated',den.shape)
for i,a in enumerate(den): rotated[i]=rotate_image(a,ROTATION_DEG)
save_stage('06_rotated',dict(angle_deg=ROTATION_DEG,reshape=False,interpolation='bilinear',padding='NaN'),'transmission')

## 7 · GDL ROI preview with pixel grid
Set an ROI inside the GDL using the rotated-image coordinate axes. Grid spacing and projection number are adjustable. Rerun freely, then approve in the following cell. Avoid padded corners, moving boundaries and non-GDL material. This ROI has one fixed geometry across time.

In [ ]:
GDL_ROI = (103,120,465,1190)                  # Replace with (x0,x1,y0,y1); None previews full image
# GDL_ROI = (0,200,0,1600)  
GRID_STEP = 50
PREVIEW_PROJECTION = 1

FONT_SIZE = 22


a=selected('06_rotated',PREVIEW_PROJECTION)
GDL_ROI=check_box(GDL_ROI,a.shape)
if GRID_STEP<=0: raise ValueError('GRID_STEP must be positive')
fig,axs=plt.subplots(1,2,figsize=(20,50),layout='constrained')

image_panel(
    axs[0], a, 'GDL ROI · rotated coordinates',
    GDL_ROI, grid=True
)

GRID_STEP = 20       # Smaller spacing = more ticks and grid lines
GRID_LINEWIDTH = 1.2 # Larger value = thicker lines

axs[0].set_xticks(np.arange(0, a.shape[1], GRID_STEP))
axs[0].set_yticks(np.arange(0, a.shape[0], GRID_STEP))

axs[0].grid(
    True,
    linewidth=GRID_LINEWIDTH,
    alpha=0.7,
    linestyle=':',
    color='#377eb8'
)


axs[0].set_xticks(np.arange(0,a.shape[1],GRID_STEP)); axs[0].set_yticks(np.arange(0,a.shape[0],GRID_STEP))


image_panel(
    axs[1], crop(a,GDL_ROI), 'ROI preview · local coordinates',
    grid=True
)

GRID_STEP = 20       # Smaller spacing = more ticks and grid lines
GRID_LINEWIDTH = 1.2 # Larger value = thicker lines

axs[0].set_xticks(np.arange(0, a.shape[1], GRID_STEP))
axs[0].set_yticks(np.arange(0, a.shape[0], GRID_STEP))

axs[0].grid(
    True,
    linewidth=GRID_LINEWIDTH,
    alpha=0.7,
    linestyle=':',
    color='#377eb8'
)


savefig(fig,'GDL_ROI_preview')
GDL_PREVIEW_SETTINGS=GDL_ROI

In [ ]:
APPROVE_GDL_ROI = False
if not APPROVE_GDL_ROI: raise RuntimeError('Inspect GDL ROI preview, then approve')
if GDL_PREVIEW_SETTINGS!=GDL_ROI: raise RuntimeError('Rerun ROI preview')
roi=alloc('07_GDL_transmission',(len(rotated),*crop(rotated[0],GDL_ROI).shape))
for i,a in enumerate(rotated): roi[i]=crop(a,GDL_ROI)
save_stage('07_GDL_transmission',dict(roi=GDL_ROI,coordinate_system='rotated initial crop'),'transmission')

## 8 · Attenuation and subtraction of the dry image
Transmission ≤0 is invalid for log and becomes NaN. T>1 is retained (negative attenuation) for diagnosis. No epsilon is inserted into the logarithm to manufacture a finite signal. A **common valid mask** across the entire time series ensures mean/SD curves compare the same pixels. The dry reference must actually be dry and representative; all projections need consistent exposure/calibration and alignment.

In [ ]:
MIN_TRANSMISSION = 0.0
if MIN_TRANSMISSION<0: raise ValueError('MIN_TRANSMISSION must be >=0')
if not 1<=REFERENCE_PROJECTION<=len(roi): raise ValueError('Invalid dry projection number')
common=np.all(np.isfinite(roi)&(roi>MIN_TRANSMISSION),axis=0)
if not common.any(): raise ValueError('No common valid ROI pixels')
print('Common valid ROI fraction:',float(common.mean()))
if common.mean()<.9: warnings.warn('More than 10% excluded: inspect dose / ROI / rotation / detector mask')
tifffile.imwrite(RUN/'common_valid_ROI_mask.tif',common.astype(np.uint8),metadata=None)
attenuation=alloc('08_attenuation',roi.shape)
delta=alloc('09_delta_attenuation',roi.shape)
for i,a in enumerate(roi):
    attenuation[i]=np.nan
    attenuation[i][common]=-np.log(a[common])
for i,a in enumerate(attenuation): delta[i]=a-attenuation[REFERENCE_PROJECTION-1]
save_stage('08_attenuation',dict(formula='-ln(T)',min_transmission=MIN_TRANSMISSION,common_valid_mask=True),'dimensionless')
save_stage('09_delta_attenuation',dict(formula='A(t)-A(dry)',reference_projection=REFERENCE_PROJECTION,
                                     reference_source=manifest.iloc[REFERENCE_PROJECTION-1].source_file),'dimensionless')

history=manifest.copy()
for label,stack in [('transmission',roi),('attenuation',attenuation),('delta_A',delta)]:
    history[label+'_mean']=[float(np.mean(a[common])) for a in stack]
    history[label+'_sd']=[float(np.std(a[common])) for a in stack]
history.to_csv(RUN/'ROI_time_history.csv',index=False)
fig,axs=plt.subplots(3,1,figsize=(13,14),sharex=True,layout='constrained')
for j,(ax,label,ylab) in enumerate(zip(axs,['transmission','attenuation','delta_A'],['Transmission','−ln(T)','Δ attenuation'])):
    mean=history[label+'_mean'].to_numpy(); sd=history[label+'_sd'].to_numpy(); t=history.elapsed_s.to_numpy()
    ax.plot(t,mean,color=SET3[j+3],marker='o',ms=3)
    ax.fill_between(t,mean-sd,mean+sd,color=SET3[j+3],alpha=.35,label='Spatial mean ± SD')
    ax.set_ylabel(ylab); ax.legend(frameon=False); ax.grid(alpha=.2)
axs[-1].set_xlabel('Elapsed time from first exposure (s)'); savefig(fig,'ROI_time_history')

## 9 · Search for the highest neighbourhood-averaged water signal
For every frame, evaluate the mean ΔA in each fully valid `PEAK_WINDOW × PEAK_WINDOW` neighbourhood. Select the largest neighbourhood mean, not the brightest pixel. A temporal median of these maps (odd `PEAK_TEMPORAL_WINDOW`, default 1, disabled) rejects isolated one-frame bursts at a fixed position. This is a persistence criterion, not a time-weighted smoother; irregular time gaps should be reviewed. The default 1 searches instantaneous neighbourhood maxima, including the first and last frames. With values ≥3, only complete centred temporal windows are eligible, so the first/last half-window frames are excluded; this can miss an end-of-run maximum. Each candidate records its time, ROI-local location, patch SD, median and support fraction. The global maximum still has selection bias; inspect it before using it as a calibration.

`PEAK_ROI` optionally restricts where the search is allowed (ROI-local coordinates), e.g. excluding edges; the entire candidate patch must fit inside it. All pixels are still normalized by one scalar. Try several reasonable window sizes; a result strongly dependent on window size is not a stable calibration.

In [ ]:
PEAK_WINDOW = 3
PEAK_TEMPORAL_WINDOW = 1
PEAK_ROI = None
PEAK_SUPPORT_FRACTION = 0.8       # Report patch fraction >= 80% of its mean
TOP_CANDIDATES = 20

if PEAK_WINDOW<3 or PEAK_WINDOW%2!=1: raise ValueError('PEAK_WINDOW must be odd and >=3')
if PEAK_TEMPORAL_WINDOW<1 or PEAK_TEMPORAL_WINDOW%2!=1 or PEAK_TEMPORAL_WINDOW>len(delta):
    raise ValueError('Temporal window must be odd, >=1 and <= number of frames')
if not 0<PEAK_SUPPORT_FRACTION<=1: raise ValueError('Support fraction must be in (0,1]')
search_box=check_box(PEAK_ROI,delta.shape[1:]); x0,x1,y0,y1=search_box
if min(x1-x0,y1-y0)<PEAK_WINDOW: raise ValueError('Peak window larger than search ROI')
search_mask=np.zeros_like(common); search_mask[y0:y1,x0:x1]=True
allowed=ndi.minimum_filter((common&search_mask).astype(np.uint8),size=PEAK_WINDOW,mode='constant',cval=0).astype(bool)
if not allowed.any(): raise ValueError('No fully valid search neighbourhood')
local_maps=alloc('diagnostic_local_means',delta.shape)
for i,a in enumerate(delta):
    m=ndi.uniform_filter(np.where(np.isfinite(a),a,0).astype(np.float64),size=PEAK_WINDOW,mode='constant')
    m[~allowed]=np.nan; local_maps[i]=m
candidates=[]; half=PEAK_TEMPORAL_WINDOW//2; radius=PEAK_WINDOW//2
for i in range(half,len(delta)-half):
    score=np.median(local_maps[i-half:i+half+1],axis=0)
    yy,xx=np.unravel_index(np.nanargmax(score),score.shape)
    patch=delta[i,yy-radius:yy+radius+1,xx-radius:xx+radius+1]
    candidates.append(dict(projection=i+1,timestamp_utc=manifest.iloc[i].timestamp_utc,
        elapsed_s=manifest.iloc[i].elapsed_s,x_roi=int(xx),y_roi=int(yy),
        score=float(score[yy,xx]),instantaneous_patch_mean=float(np.mean(patch)),
        patch_sd=float(np.std(patch)),patch_median=float(np.median(patch)),
        support_fraction=float(np.mean(patch>=PEAK_SUPPORT_FRACTION*np.mean(patch))),
        support_start_utc=manifest.iloc[i-half].timestamp_utc,support_end_utc=manifest.iloc[i+half].timestamp_utc))
peak_table=pd.DataFrame(candidates)
peak_table.to_csv(RUN/'peak_neighbourhood_per_frame.csv',index=False)
ranked=peak_table.sort_values('score',ascending=False)
ranked.head(TOP_CANDIDATES).to_csv(RUN/'top_peak_candidates.csv',index=False)
best=ranked.iloc[0]; OBSERVED_MAX=float(best.score)
if not np.isfinite(OBSERVED_MAX) or OBSERVED_MAX<=0: raise ValueError('No positive maximum water-change signal')
pi=int(best.projection)-1; yy=int(best.y_roi); xx=int(best.x_roi)
peak_box=(xx-radius,xx+radius+1,yy-radius,yy+radius+1)
score=np.median(local_maps[pi-half:pi+half+1],axis=0)
fig,axs=plt.subplots(2,2,figsize=(17,13),layout='constrained')
m=image_panel(axs[0,0],delta[pi],f'ΔA · projection {pi+1}',peak_box); fig.colorbar(m,ax=axs[0,0])
m=image_panel(axs[0,1],score,'Neighbourhood score',peak_box); fig.colorbar(m,ax=axs[0,1])
axs[1,0].plot(peak_table.elapsed_s,peak_table.score,color=SET3[3],label='Frame maximum score')
axs[1,0].axvline(best.elapsed_s,color='0.3',ls='--'); axs[1,0].set(xlabel='Elapsed time (s)',ylabel='Neighbourhood ΔA'); axs[1,0].legend(frameon=False)
fixed_trace=np.array([np.mean(crop(a,peak_box)) for a in delta])
fixed_sd=np.array([np.std(crop(a,peak_box)) for a in delta])
t=manifest.elapsed_s.to_numpy()
axs[1,1].plot(t,fixed_trace,color=SET3[4]); axs[1,1].fill_between(t,fixed_trace-fixed_sd,fixed_trace+fixed_sd,color=SET3[4],alpha=.35)
axs[1,1].set(xlabel='Elapsed time (s)',ylabel='Selected patch ΔA',title='Fixed patch history · mean ± SD')
savefig(fig,'maximum_diagnostics')
manifest.assign(patch_mean=fixed_trace,patch_sd=fixed_sd).to_csv(RUN/'selected_peak_patch_history.csv',index=False)
display(ranked.head(TOP_CANDIDATES)); print('Proposed 100% reference ΔA:',OBSERVED_MAX)
PEAK_PREVIEW_SETTINGS=(PEAK_WINDOW,PEAK_TEMPORAL_WINDOW,PEAK_ROI,OBSERVED_MAX)


## 10 · Approve the scale and save relative saturation
By default use the observed maximum above. To use an independently measured saturated reference, set `SATURATION_REFERENCE_DELTA_A` explicitly and document its origin. This is one scalar calibration; a thickness-varying GDL may require a spatial calibration map instead. Raw saturation is not clipped; out-of-range fractions are reported.

In [ ]:
def save_stage(name, details, unit='a.u.'):
    a = stages[name]
    d = RUN / name
    d.mkdir(exist_ok=True)

    stage_metadata[name] = dict(details=details, unit=unit)
    table = manifest.copy()
    rows = []

    for i, frame in enumerate(a):
        fname = f'proj_{i+1:06d}.tif'

        meta = dict(
            manifest.iloc[i].to_dict(),
            stage=name,
            unit=unit,
            details=details,
            upstream={
                k: v for k, v in stage_metadata.items() if k < name
            },
            pixel_size_um=PIXEL_SIZE_UM,
            full_original_headers='../original_headers',
            invalid='NaN'
        )

        tifffile.imwrite(
            d / fname,
            np.asarray(frame, np.float32),
            photometric='minisblack',
            description=json.dumps(meta, default=str),
            metadata=None
        )
        rows.append(stats(frame))

    table['tiff_file'] = [
        f'proj_{i+1:06d}.tif' for i in range(len(a))
    ]
    table.to_csv(d / 'timestamps.csv', index=False)

    table = pd.concat(
        [table.reset_index(drop=True), pd.DataFrame(rows)],
        axis=1
    )
    table.to_csv(d / 'statistics.csv', index=False)

    (RUN / 'processing.json').write_text(
        json.dumps(stage_metadata, indent=2, default=str)
    )

    print('Saved', name, a.shape)
    return table

In [ ]:
# ── User settings ──────────────────────────────────────────────
IMAGE_INDEX = 1         # Zero-based display index: 1 is projection 2; reference 0 stays exported
IMAGE_VMIN = 0        # Minimum displayed saturation (%)
IMAGE_VMAX = 100      # Maximum displayed saturation (%)

APPROVE_SATURATION_REFERENCE = False
SATURATION_REFERENCE_DELTA_A = None
REFERENCE_NOTE = (
    'Observed neighbourhood maximum; relative signal scale (temporal window recorded separately)'
)

# ── Validate settings before processing ────────────────────────
if not APPROVE_SATURATION_REFERENCE:
    raise RuntimeError('Inspect peak diagnostics, then approve reference')

if PEAK_PREVIEW_SETTINGS != (
    PEAK_WINDOW, PEAK_TEMPORAL_WINDOW, PEAK_ROI, OBSERVED_MAX
):
    raise RuntimeError('Rerun peak diagnostics')

scale = float(
    OBSERVED_MAX
    if SATURATION_REFERENCE_DELTA_A is None
    else SATURATION_REFERENCE_DELTA_A
)

if not np.isfinite(scale) or scale <= 0:
    raise ValueError('Saturation reference must be finite and positive')

if len(delta) < 2:
    raise ValueError('At least two images are needed to exclude the first projection')

if (
    isinstance(IMAGE_INDEX, (bool, np.bool_))
    or not isinstance(IMAGE_INDEX, (int, np.integer))
    or not 1 <= IMAGE_INDEX < len(delta)
):
    raise ValueError(
        f'IMAGE_INDEX must be an integer between 1 and {len(delta) - 1}. '
        'Index 0 is the excluded first projection.'
    )

if (
    not np.isfinite(IMAGE_VMIN)
    or not np.isfinite(IMAGE_VMAX)
    or IMAGE_VMIN >= IMAGE_VMAX
):
    raise ValueError(
        'IMAGE_VMIN and IMAGE_VMAX must be finite, with IMAGE_VMIN < IMAGE_VMAX'
    )

common_mask = np.asarray(common, dtype=bool)
if common_mask.shape != delta.shape[1:] or not common_mask.any():
    raise ValueError('common must be a nonempty mask matching the image shape')

# ── Calculate and save saturation for all projections ──────────
saturation = alloc('10_relative_saturation', delta.shape)

for i, a in enumerate(delta):
    saturation[i] = 100 * a / scale

sat_stats = save_stage(
    '10_relative_saturation',
    dict(
        formula='100*delta_A/reference_delta_A',
        reference_delta_A=scale,
        reference_note=REFERENCE_NOTE,
        peak_window=PEAK_WINDOW,
        temporal_window=PEAK_TEMPORAL_WINDOW,
        peak_roi=PEAK_ROI,
        best_candidate=best.to_dict(),
        palette=SAT_COLORS,
        clipped=False
    ),
    '%'
)

below_0 = []
above_100 = []

for a in saturation:
    values = a[common_mask]
    values = values[np.isfinite(values)]
    below_0.append(float(np.mean(values < 0)) if values.size else np.nan)
    above_100.append(float(np.mean(values > 100)) if values.size else np.nan)

sat_stats['below_0_fraction'] = below_0
sat_stats['above_100_fraction'] = above_100
sat_stats.to_csv(RUN / 'saturation_time_history.csv', index=False)

# ── Time-history plot: exclude first projection, no title ──────
t = sat_stats['elapsed_s'].to_numpy()
m = sat_stats['mean'].to_numpy()
s = sat_stats['sd'].to_numpy()

fig, ax = plt.subplots(figsize=(13, 6))
ax.plot(t[1:], m[1:], color=SET3[0])
ax.fill_between(
    t[1:],
    m[1:] - s[1:],
    m[1:] + s[1:],
    color=SET3[0],
    alpha=0.35,
    label='Spatial mean ± SD'
)
ax.set(
    xlabel='Elapsed time (s)',
    ylabel='Relative saturation (%)'
)
ax.legend(frameon=False)
ax.grid(alpha=0.2)
fig.tight_layout()
savefig(fig, 'saturation_time_history')
plt.show()

# ── Selected image: bottom-left origin, manual limits, no title ──
a = saturation[IMAGE_INDEX]
display_image = np.ma.array(
    a,
    mask=(~common_mask) | (~np.isfinite(a))
)

fig, ax = plt.subplots(figsize=(15, 25))
im = ax.imshow(
    display_image,
    cmap=SAT_CMAP,
    vmin=IMAGE_VMIN,
    vmax=IMAGE_VMAX,
    origin='lower',
    interpolation='nearest'
)
ax.set(
    xlabel='X (pixels)',
    ylabel='Y (pixels)'
)
# fig.colorbar(im, ax=ax, label='Relative saturation (%)')
fig.colorbar(
    im,
    ax=ax,
    label='Relative saturation (%)',
    fraction=0.025,
    pad=0.03,
    aspect=50
)

fig.tight_layout()
savefig(fig, f'saturation_image_{IMAGE_INDEX:05d}')
plt.show()

print('Finished. Saturation TIFF folder:', RUN / '10_relative_saturation')

## 11 · Independent saturation TIFF viewer 
**This last cell works in a fresh kernel** after installing numpy, pandas, matplotlib and tifffile. Set the saturation folder and projection number. It reads only the selected TIFF using `timestamps.csv` (or natural filename order if no manifest). Horizontal and vertical line groups have separate plots. The central image has the supplied 0–100% colour scale, a right marginal mean over X with SD over X, and a bottom marginal mean over Y with SD over Y. Colour scale is not a physical length scale; set `VIEW_PIXEL_SIZE_UM` if you also want a length bar and physical axes. Quantitative profiles remain unclipped. CSVs and PNG/PDF plots go into `standalone_profiles` in that folder. The saved TIFF's full processing metadata includes the palette and pixel size.

In [ ]:
from pathlib import Path
import re, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.patheffects as pe
import tifffile

import os
REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
_default_run = globals().get('RUN', REPO_ROOT / 'outputs' / 'YOUR_RUN')
SATURATION_FOLDER = Path(os.environ.get(
    'GDL_SATURATION_DIR', str(_default_run / '10_relative_saturation')))
VIEW_PROJECTION = 2               # 1-based projection ID from manifest; choose for your data
VIEW_HORIZONTAL_Y = [100,200,300,400,500,600,700]          # Pixel row coordinates (always pixels, even with physical axes)
VIEW_VERTICAL_X = [1,4,10,14]              # Pixel column coordinates



VIEW_LINE_HALF_WIDTH = 0
VIEW_PIXEL_SIZE_UM = None       # None: use TIFF metadata if available, otherwise pixels
VIEW_FONT_SIZE = 20
VIEW_SCALE_BAR_UM = None        # None: sensible automatic length if pixel size is known
VIEW_FALLBACK_COLORS = ['#c07127', '#bd6e24', '#d78c3f', '#d2883e', '#d2893d', '#d5893e', '#e3a764', '#e9ac67', '#f4cf86', '#fdd78b', '#f9d488', '#fbe9a5', '#f9e7a3', '#fceaa6', '#f9e5a2', '#fefac0', '#fff5bb', '#fef8bc', '#ffffff', '#fefefe', '#fefefe', '#fefefe', '#d7fed7', '#d6fed6', '#d2fdd2', '#9ff7b9', '#a0f8ba', '#9ef6b7', '#9cf2b5', '#53bfa1', '#55c5a7', '#6facc9', '#75b4d3', '#71afcd', '#70afcd', '#0076ba', '#0074b7', '#0075b8', '#006dad', '#003559']

plt.rcParams.update({'font.size':VIEW_FONT_SIZE,'axes.labelsize':VIEW_FONT_SIZE,
                     'xtick.labelsize':VIEW_FONT_SIZE-2,'ytick.labelsize':VIEW_FONT_SIZE-2,
                     'legend.fontsize':VIEW_FONT_SIZE-4,'lines.linewidth':2.7,
                     'pdf.fonttype':42,'ps.fonttype':42})
output=SATURATION_FOLDER/'standalone_profiles'; output.mkdir(parents=True,exist_ok=True)
manifest_path=SATURATION_FOLDER/'timestamps.csv'
if manifest_path.exists():
    view_manifest=pd.read_csv(manifest_path)
    matches=view_manifest.loc[view_manifest.projection==VIEW_PROJECTION]
    if len(matches)!=1: raise ValueError('Projection ID missing or duplicated in manifest')
    record=matches.iloc[0].to_dict(); image_path=SATURATION_FOLDER/record['tiff_file']
else:
    paths=sorted([*SATURATION_FOLDER.glob('*.tif'),*SATURATION_FOLDER.glob('*.tiff')],
                 key=lambda p:[int(v) if v.isdigit() else v.lower() for v in re.split(r'(\d+)',p.name)])
    if not 1<=VIEW_PROJECTION<=len(paths): raise ValueError('Projection outside file list')
    image_path=paths[VIEW_PROJECTION-1]; record={'projection':VIEW_PROJECTION}
with tifffile.TiffFile(image_path) as tf:
    view_image=tf.asarray().astype(np.float32)
    try: image_meta=json.loads(tf.pages[0].description or '{}')
    except (ValueError,TypeError): image_meta={}
if view_image.ndim!=2: raise ValueError('Expected one 2D saturation image per TIFF')
if image_meta.get('unit','%')!='%': raise ValueError('Selected TIFF is not a saturation percentage image')
if not np.isfinite(view_image).any(): raise ValueError('Selected image has no valid data')
view_time=record.get('timestamp_utc',image_meta.get('timestamp_utc','timestamp unavailable'))
view_elapsed=record.get('elapsed_s',image_meta.get('elapsed_s',np.nan))
cm=LinearSegmentedColormap.from_list('GDL_relative_saturation',image_meta.get('details',{}).get('palette',VIEW_FALLBACK_COLORS),N=256)
cm.set_bad('#777777'); cols=list(plt.get_cmap('Set3').colors)
px=VIEW_PIXEL_SIZE_UM if VIEW_PIXEL_SIZE_UM is not None else image_meta.get('pixel_size_um')
if px is not None and (not np.isfinite(px) or px<=0): raise ValueError('Pixel size must be positive')
factor=1. if px is None else float(px); unit='pixel' if px is None else 'µm'
h,w=view_image.shape; x=np.arange(w)*factor; y=np.arange(h)*factor
extent = (-.5*factor, (w-.5)*factor,-.5*factor, (h-.5)*factor)
right_mean=np.nanmean(view_image,axis=1); right_sd=np.nanstd(view_image,axis=1)
bottom_mean=np.nanmean(view_image,axis=0); bottom_sd=np.nanstd(view_image,axis=0)


fig = plt.figure(figsize=(15, 25), layout='constrained')
gs = fig.add_gridspec(2, 2,width_ratios=[4, 1.5],height_ratios=[4, 1.5])
ax = fig.add_subplot(gs[0, 0])
ar = fig.add_subplot(gs[0, 1], sharey=ax)
ab = fig.add_subplot(gs[1, 0], sharex=ax)
im = ax.imshow(view_image,cmap=cm,vmin=0,vmax=100,origin='lower',extent=extent,interpolation='nearest')
ax.set(xlabel=f'X ({unit})', ylabel=f'Y ({unit})')
ar.plot(right_mean,y,color=cols[0]); ar.fill_betweenx(y,right_mean-right_sd,right_mean+right_sd,color=cols[0],alpha=.4)
ar.set(xlabel='Mean ± SD (%)',title='Mean over X'); ar.tick_params(labelleft=False)

ab.plot(x,bottom_mean,color=cols[0]); ab.fill_between(x,bottom_mean-bottom_sd,bottom_mean+bottom_sd,color=cols[0],alpha=.4)
ab.set(xlabel=f'X ({unit})',ylabel='Mean ± SD (%)',title='Mean over Y')
    

if px is not None:
    length=VIEW_SCALE_BAR_UM if VIEW_SCALE_BAR_UM is not None else max(factor,round(w*factor/5,2))
    if not 0<length<w*factor*.8: raise ValueError('Scale bar too long for image')
    xstart=w*factor*.07; ybar=h*factor*.92
    ax.plot([xstart,xstart+length],[ybar,ybar],color='black',lw=5,
            path_effects=[pe.Stroke(linewidth=8,foreground='white'),pe.Normal()])
    ax.text(xstart+length/2,ybar-h*factor*.03,f'{length:g} µm',ha='center',color='black',
            path_effects=[pe.withStroke(linewidth=3,foreground='white')])
profile_fig,profile_axes=plt.subplots(1,2,figsize=(17,6),layout='constrained')
profile_rows=[]
if int(VIEW_LINE_HALF_WIDTH)!=VIEW_LINE_HALF_WIDTH or VIEW_LINE_HALF_WIDTH<0: raise ValueError('Invalid half-width')
band_radius=int(VIEW_LINE_HALF_WIDTH)
for orientation,positions,pax in [('horizontal',VIEW_HORIZONTAL_Y,profile_axes[0]),('vertical',VIEW_VERTICAL_X,profile_axes[1])]:
    for j,pos in enumerate(positions):
        bound=h if orientation=='horizontal' else w
        if int(pos)!=pos or not band_radius<=pos<bound-band_radius: raise ValueError(f'{orientation} band at {pos} outside image')
        pos=int(pos); color=cols[j%12]
        band=view_image[pos-band_radius:pos+band_radius+1,:] if orientation=='horizontal' else view_image[:,pos-band_radius:pos+band_radius+1]
        axis=0 if orientation=='horizontal' else 1
        m=np.nanmean(band,axis=axis); s=np.nanstd(band,axis=axis); coord=x if orientation=='horizontal' else y
        pax.plot(coord,m,color=color,label=f'{"y" if orientation=="horizontal" else "x"}={pos} px',
                 path_effects=[pe.Stroke(linewidth=3.5,foreground='0.35'),pe.Normal()])
        if band_radius: pax.fill_between(coord,m-s,m+s,color=color,alpha=.3)
        (ax.axhline if orientation=='horizontal' else ax.axvline)(pos*factor,color=color,ls='--',lw=1.8)
        profile_rows.extend(dict(projection=VIEW_PROJECTION,timestamp_utc=view_time,elapsed_s=view_elapsed,
            orientation=orientation,line_pixel=pos,coordinate_pixel=k,coordinate=float(c),coordinate_unit=unit,
            mean_percent=float(v),sd_percent=float(sd)) for k,(c,v,sd) in enumerate(zip(coord,m,s)))
    pax.set(xlabel=f'{"X" if orientation=="horizontal" else "Y"} ({unit})',ylabel='Relative saturation (%)',title=orientation.title()+' profiles')
    pax.grid(alpha=.2)
    if positions: pax.legend(frameon=False)
for orientation,coord,m,s in [('right',y,right_mean,right_sd),('bottom',x,bottom_mean,bottom_sd)]:
    pd.DataFrame(dict(projection=VIEW_PROJECTION,timestamp_utc=view_time,elapsed_s=view_elapsed,
                     coordinate_pixel=np.arange(len(coord)),coordinate=coord,coordinate_unit=unit,
                     mean_percent=m,sd_percent=s)).to_csv(output/f'projection_{VIEW_PROJECTION:06d}_{orientation}_mean_SD.csv',index=False)
pd.DataFrame(profile_rows).to_csv(output/f'projection_{VIEW_PROJECTION:06d}_line_profiles.csv',index=False)



cax = ax.inset_axes([1.3, 0.0, 0.45, 1.0])
fig.colorbar(im,cax=cax,label='Relative saturation (%)',extend='neither')

fig.canvas.draw()  # Resolve the image's actual displayed bounds.

map_position = ax.get_position()
right_position = ar.get_position()
bottom_position = ab.get_position()

# Freeze the layout so saving does not undo the adjustment.
fig.set_layout_engine('none')

ab.set_position([
    map_position.x0,                         # Left edge: map
    bottom_position.y0,                      # Keep vertical position
    right_position.x1 - map_position.x0,      # Right edge: Mean over X
    bottom_position.height                   # Keep height
])



for name,f in [('map_marginals',fig),('line_profiles',profile_fig)]:
    for extension in ['png','pdf']: f.savefig(output/f'projection_{VIEW_PROJECTION:06d}_{name}.{extension}',dpi=300,bbox_inches='tight')
plt.show()
print('Source:',image_path.name,'| timestamp:',view_time,'| elapsed s:',view_elapsed)
print('Finite range retained:',float(np.nanmin(view_image)),float(np.nanmax(view_image)),'%')




# ── Additional settings ────────────────────────────────────────
IMAGE_VMIN = 0
IMAGE_VMAX = 100
IMAGE_SHOW_COLORBAR = True
TIME_HISTORY_FILE = SATURATION_FOLDER.parent / 'saturation_time_history.csv'

if (
    not np.isfinite(IMAGE_VMIN)
    or not np.isfinite(IMAGE_VMAX)
    or IMAGE_VMIN >= IMAGE_VMAX
):
    raise ValueError(
        'IMAGE_VMIN and IMAGE_VMAX must be finite, with IMAGE_VMIN < IMAGE_VMAX'
    )

# Apply the same display limits to the existing map.
# Display limits do not clip the underlying saturation values.
im.set_clim(IMAGE_VMIN, IMAGE_VMAX)

for extension in ['png', 'pdf']:
    fig.savefig(
        output / f'projection_{VIEW_PROJECTION:06d}_map_marginals.{extension}',
        dpi=300,
        bbox_inches='tight'
    )

# ── Load and plot saved time history ───────────────────────────
# These TIFFs already contain saturation percentages.
# No reference approval or saturation recalculation is needed.
if TIME_HISTORY_FILE.exists():
    sat_stats = pd.read_csv(TIME_HISTORY_FILE)

    required_columns = {'elapsed_s', 'mean', 'sd'}
    missing_columns = required_columns - set(sat_stats.columns)
    if missing_columns:
        raise ValueError(
            f'Time-history CSV is missing columns: {sorted(missing_columns)}'
        )

    # Match the original processing plot: exclude the first saved row.
    history = sat_stats.iloc[1:]

    t = history['elapsed_s'].to_numpy(dtype=float)
    m = history['mean'].to_numpy(dtype=float)
    s = history['sd'].to_numpy(dtype=float)

    history_fig, history_ax = plt.subplots(figsize=(13, 6))
    history_ax.plot(t, m, color=cols[0])
    history_ax.fill_between(
        t,
        m - s,
        m + s,
        color=cols[0],
        alpha=0.35,
        label='Spatial mean ± SD'
    )
    history_ax.set(
        xlabel='Elapsed time (s)',
        ylabel='Relative saturation (%)'
    )
    history_ax.legend(frameon=False)
    history_ax.grid(alpha=0.2)
    history_fig.tight_layout()

    for extension in ['png', 'pdf']:
        history_fig.savefig(
            output / f'saturation_time_history.{extension}',
            dpi=300,
            bbox_inches='tight'
        )
else:
    print(
        'Time-history plot skipped: saved CSV not found at',
        TIME_HISTORY_FILE
    )

# ── Separate selected image: bottom-left origin, no title ──────
display_image = np.ma.masked_invalid(view_image)

image_fig, image_ax = plt.subplots(figsize=(15, 25))
image_im = image_ax.imshow(
    display_image,
    cmap=cm,
    vmin=IMAGE_VMIN,
    vmax=IMAGE_VMAX,
    origin='lower',
    extent=extent,
    interpolation='nearest'
)
image_ax.set(
    xlabel=f'X ({unit})',
    ylabel=f'Y ({unit})'
)

if IMAGE_SHOW_COLORBAR:
    image_fig.colorbar(
        image_im,
        ax=image_ax,
        label='Relative saturation (%)',
        fraction=0.025,
        pad=0.03,
        aspect=50
    )

image_fig.tight_layout()

for extension in ['png', 'pdf']:
    image_fig.savefig(
        output / f'projection_{VIEW_PROJECTION:06d}_saturation_image.{extension}',
        dpi=300,
        bbox_inches='tight'
    )

plt.show()
print('Plots saved to:', output)

### References and practical notes
- Original normalization: supplied `Dose_Normalization_LineProfile_Pranav.ipynb`, median dose-window implementation. This version deliberately rejects invalid dose estimates and preserves invalid pixels rather than substituting zeros or epsilon dose factors.
- [Astropy time handling](https://docs.astropy.org/en/stable/time/) and [FITS time coordinates](https://www.atnf.csiro.au/computing/software/wcs/tcs.pdf).
- [Official BM3D software and papers](https://webpages.tuni.fi/foi/GCF-BM3D/index.html). Check the package's licence for your intended use.
- Before interpreting uptake: inspect dose stability, residual images, invalid-pixel fraction, fixed-patch history and registration. This notebook does not automatically register frames or correct scattering, beam hardening, changing sample thickness or motion.
- Computational cost: float32 storage is `4 × N × H × W` bytes per full stack. The workflow intentionally retains stages for inspection; use `STORAGE='memmap'` for larger datasets and allow ample disk space for caches plus TIFF exports. You may delete `cache` only after closing the notebook and releasing memmaps; saved TIFFs remain usable.
- The included tests use synthetic data. Validate ROIs, timestamps, spike thresholds and BM3D sigma against experimental images and saved processing metadata before publication. See `docs/author_checks.md`.